In [23]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv("/Users/bytedance/Documents/Mauriciio/Jupyter/Metallica/.env")

LASTFM_API_KEY = os.getenv("LASTFM_API_KEY")

BASE_URL = "http://ws.audioscrobbler.com/2.0/"

artists = ["Metallica", "Megadeth", "Slayer", "Anthrax"]

In [24]:
## Getting popular tracks LAST.FM
all_tracks = []

for artist in artists:
    params = {
        "method": "artist.gettoptracks",
        "artist": artist,
        "api_key": LASTFM_API_KEY,
        "format": "json",
        "limit": 10
    }
    
    response = requests.get(BASE_URL, params=params)
    data = response.json()
    
    tracks = data["toptracks"]["track"]
    
    for rank, track in enumerate(tracks, start=1):
        all_tracks.append({
            "artist": artist,
            "rank": rank,
            "track_name": track["name"],
            "listeners": int(track["listeners"]),
            "playcount": int(track["playcount"]),
            "lastfm_url": track["url"]
        })

top_tracks_df = pd.DataFrame(all_tracks)

top_tracks_df

,artist,rank,track_name,listeners,playcount,lastfm_url
0,Metallica,1,Enter Sandman,1888381,18686946,https://www.last.fm/music/Metallica/_/Enter+Sa...
1,Metallica,2,Nothing Else Matters,1705979,17073053,https://www.last.fm/music/Metallica/_/Nothing+...
2,Metallica,3,Master of Puppets,1650172,16766790,https://www.last.fm/music/Metallica/_/Master+o...
3,Metallica,4,One,1397236,14026914,https://www.last.fm/music/Metallica/_/One
4,Metallica,5,The Unforgiven,1178284,10814541,https://www.last.fm/music/Metallica/_/The+Unfo...
5,Metallica,6,Sad but True,1034844,9030880,https://www.last.fm/music/Metallica/_/Sad+but+...
6,Metallica,7,Fuel,1020088,9031571,https://www.last.fm/music/Metallica/_/Fuel
7,Metallica,8,Battery,915605,7978445,https://www.last.fm/music/Metallica/_/Battery
8,Metallica,9,Whiskey in the Jar,908739,7994202,https://www.last.fm/music/Metallica/_/Whiskey+...
9,Metallica,10,Wherever I May Roam,828139,6542129,https://www.last.fm/music/Metallica/_/Wherever...


In [25]:
## Saving data to csv
top_tracks_df.to_csv("lastfm_top_tracks_big4.csv", index=False)

In [6]:
## Music Brainz
!pip install musicbrainzngs pandas

In [26]:
## Connect to MB
import musicbrainzngs
import pandas as pd
import time

musicbrainzngs.set_useragent(
    "BigFourThrashDashboard",
    "0.1",
    "macob39@gmail.com"
)

In [27]:
## Getting big four data
artists = ["Metallica", "Megadeth", "Slayer", "Anthrax"]

all_albums_list = []
all_tracks_list = []

for artist_name in artists:
    print(f"Processing {artist_name}...")
    
    artist_id, clean_artist_name = get_artist_id(artist_name)
    
    albums_df = get_studio_release_groups(artist_id, clean_artist_name)
    
    track_rows = []
    
    for _, album in albums_df.iterrows():
        tracks = get_tracks_from_release_group(album["release_group_id"])
        
        for track in tracks:
            track_rows.append({
                "artist": album["artist"],
                "album_name": album["album_name"],
                "first_release_date": album["first_release_date"],
                "release_group_id": album["release_group_id"],
                **track
            })
    
    tracks_df = pd.DataFrame(track_rows)
    
    safe_name = artist_name.lower().replace(" ", "_")
    
    albums_df.to_csv(f"{safe_name}_studio_albums_musicbrainz.csv", index=False)
    tracks_df.to_csv(f"{safe_name}_studio_tracks_musicbrainz.csv", index=False)
    
    all_albums_list.append(albums_df)
    all_tracks_list.append(tracks_df)

print("Done!")

Processing Metallica...
Processing Megadeth...
Processing Slayer...
Processing Anthrax...
Done!


In [28]:
## Join data
big4_albums = pd.concat(all_albums_list, ignore_index=True)
big4_tracks = pd.concat(all_tracks_list, ignore_index=True)

big4_albums.to_csv("big4_studio_albums_musicbrainz.csv", index=False)
big4_tracks.to_csv("big4_studio_tracks_musicbrainz.csv", index=False)

big4_tracks.head()

,artist,album_name,first_release_date,release_group_id,release_id,recording_id,track_number,track_position,track_name,track_length_ms
0,Metallica,Kill ’Em All,1983-07-25,f44f4f73-a714-31a1-a4b8-bfcaaf311f50,a89e1d92-5381-4dab-ba51-733137d0e431,a620f5e7-ae9d-4089-a947-a1cee4a9435e,A1,1,Hit the Lights,257000
1,Metallica,Kill ’Em All,1983-07-25,f44f4f73-a714-31a1-a4b8-bfcaaf311f50,a89e1d92-5381-4dab-ba51-733137d0e431,04fc57f2-4dc8-4b84-801b-3699508597e8,A2,2,The Four Horsemen,432706
2,Metallica,Kill ’Em All,1983-07-25,f44f4f73-a714-31a1-a4b8-bfcaaf311f50,a89e1d92-5381-4dab-ba51-733137d0e431,65ca9607-fa5d-4b5d-b830-3fb168ac9064,A3,3,Motorbreath,188133
3,Metallica,Kill ’Em All,1983-07-25,f44f4f73-a714-31a1-a4b8-bfcaaf311f50,a89e1d92-5381-4dab-ba51-733137d0e431,321b3122-f443-4709-8098-3afba7cddca9,A4,4,Jump in the Fire,281493
4,Metallica,Kill ’Em All,1983-07-25,f44f4f73-a714-31a1-a4b8-bfcaaf311f50,a89e1d92-5381-4dab-ba51-733137d0e431,78095e21-30cf-47eb-b367-3af6ee426af5,A5,5,(Anesthesia) Pulling Teeth,254840


In [29]:
big4_albums.groupby("artist")["album_name"].nunique()
big4_tracks.groupby("artist")["track_name"].count()

artist
Anthrax      155
Megadeth     188
Metallica    154
Slayer       168
Name: track_name, dtype: int64